# 电商用户行为分析

本 Notebook 覆盖五个分析模块：

1. **数据加载与质量校验**
2. **漏斗分析**（修正分子分母口径）
3. **A/B 测试**（含 95% 置信区间）
4. **留存分析**（双口径对比 + 同期群）
5. **收入与 LTV 分析**
6. **流失预测**（早期预警 vs 常规模型）

分析逻辑抽到 `analytics/` 包中，Notebook 与 Streamlit 仪表盘共用同一份代码。

## 0. 环境准备

In [23]:
import sys
from pathlib import Path

# 让 analytics 包可被 import
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import duckdb
from IPython.display import display, Markdown

from analytics import (
    CFG,
    load_data, init_duckdb, validate_data,
    get_funnel_overall, get_funnel_by_dimension, get_funnel_all_dimensions,
    run_ab_test, ab_segment_effects, ci_overlap_check,
    get_overall_retention, get_retention_by_experiment,
    get_cohort_table, get_segment_retention, retention_dual_view,
    get_user_revenue, get_revenue_summary, get_revenue_by_experiment,
    get_cohort_revenue, get_segment_revenue, get_ltv_curve,
    build_churn_model, compare_churn_models,
    plot_retention_curves, plot_cohort_heatmap, plot_funnel,
    plot_ab_test, plot_ltv_curve, plot_churn_coefficients,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print("配置：")
print(f"  留存口径      = {CFG.retention_denominator}")
print(f"  流失标签周    = week {CFG.churn_final_week}")
print(f"  早期预警窗口  = week <= {CFG.churn_early_window}")
print(f"  常规模型窗口  = week <= {CFG.churn_regular_window}")

配置：
  留存口径      = week0_active
  流失标签周    = week 8
  早期预警窗口  = week <= 1
  常规模型窗口  = week <= 4


## 1. 数据加载与质量校验

### 1.1 加载

三张表：

- `users.csv`：用户维表（注册日期、渠道、设备、分群、实验分组等）
- `transactions.csv`：交易明细
- `retention.csv`：用户 × 周次 的活跃状态

In [24]:
users, transactions, retention = load_data()
con = init_duckdb(users, transactions, retention)

print(f"users        : {users.shape}")
print(f"transactions : {transactions.shape}")
print(f"retention    : {retention.shape}")

display(users.head())
display(transactions.head())
display(retention.head())

users        : (50000, 9)
transactions : (5277, 5)
retention    : (600000, 3)


,user_id,signup_date,acquisition_channel,country,device_type,user_segment,experiment_group,activated,purchased
0,1,2024-04-12,organic,US,desktop,price_sensitive,control,True,True
1,2,2024-06-28,referral,India,mobile,power,control,False,False
2,3,2024-04-02,paid,UK,mobile,casual,treatment,False,False
3,4,2024-01-15,organic,US,desktop,price_sensitive,treatment,False,False
4,5,2024-04-16,paid,US,mobile,price_sensitive,treatment,False,False


,transaction_id,user_id,transaction_date,plan_type,revenue
0,1,1,2024-04-19,monthly,20
1,2,16,2024-05-13,monthly,20
2,3,25,2024-09-08,monthly,20
3,4,29,2024-07-22,monthly,20
4,5,51,2024-04-27,annual,200


,user_id,week,is_active
0,1,0,1
1,1,1,1
2,1,2,1
3,1,3,1
4,1,4,1


### 1.2 数据质量校验

检查项：

| 字段 | 说明 |
|------|------|
| `duplicate_user_week` | (user_id, week) 重复行数，应为 0 |
| `users_with_missing_weeks` | 记录数不足 12 周的用户数 |
| `week0_active_users` | week 0 活跃的用户数（标准留存分母） |
| `users_not_in_retention` | users 表有但 retention 表没有的用户数 |
| `retention_users_not_in_users` | retention 表有但 users 表没有的用户数 |
| `purchased_but_not_activated` | 购买了但未激活的用户数（口径异常） |

In [25]:
report = validate_data(users, transactions, retention)
display(pd.DataFrame([report]).T.rename(columns={0: "值"}))

# 关注：users_with_missing_weeks 是否 > 0
if report["users_with_missing_weeks"] > 0:
    print(f"\n⚠️ 有 {report['users_with_missing_weeks']} 个用户记录不足 12 周")

,值
duplicate_user_week,0
users_with_missing_weeks,0
total_users_in_retention,50000
week0_active_users,46472
users_not_in_retention,0
retention_users_not_in_users,0
purchased_but_not_activated,0


## 2. 漏斗分析

### 2.1 口径说明

相比原始 SQL，修正了三点：

1. **布尔比较**：`activated = 'True'` → `CASE WHEN activated THEN ...`，避免隐式转换。
2. **除零保护**：分母加 `NULLIF(..., 0)`。
3. **分子口径自洽**：购买率分子只统计 **已激活且购买** 的用户。

In [26]:
funnel_overall = get_funnel_overall(con)
display(
    funnel_overall.style.format({
        "activation_rate": "{:.2%}",
        "purchase_rate_from_activated": "{:.2%}",
        "overall_conversion": "{:.2%}",
    })
)

plot_funnel(funnel_overall).show()

,signups,activated_users,purchasers,activation_rate,purchase_rate_from_activated,overall_conversion
0,50000,18874.000000,5277,37.75%,27.96%,10.55%


### 2.2 按维度拆分

支持 4 个维度：`acquisition_channel` / `user_segment` / `country` / `device_type`。

In [27]:
funnel_all = get_funnel_all_dimensions(con)
display(funnel_all)

# 关注：哪个维度的激活率、整体转化率有明显差异
for dim in ["acquisition_channel", "user_segment", "country", "device_type"]:
    sub = funnel_all.query("dimension == @dim")
    display(Markdown(f"**按 {CFG.dim_labels[dim]} 的漏斗**"))
    display(sub[["dimension_value", "signups", "activated_users",
                 "purchasers", "activation_rate", "overall_conversion"]])

,dimension,dimension_value,signups,activated_users,purchasers,activation_rate,purchase_rate_from_activated,overall_conversion
0,acquisition_channel,paid,20059,7556.0,2081,0.3767,0.2754,0.1037
1,acquisition_channel,referral,9912,3828.0,1108,0.3862,0.2894,0.1118
2,acquisition_channel,organic,20029,7490.0,2088,0.3740,0.2788,0.1042
3,user_segment,price_sensitive,19954,7009.0,1917,0.3513,0.2735,0.0961
4,user_segment,power,14979,6708.0,1936,0.4478,0.2886,0.1292
5,user_segment,casual,15067,5157.0,1424,0.3423,0.2761,0.0945
6,country,India,15017,5637.0,1577,0.3754,0.2798,0.1050
7,country,UK,9995,3768.0,1049,0.3770,0.2784,0.1050
8,country,US,24988,9469.0,2651,0.3789,0.2800,0.1061
9,device_type,mobile,29852,10635.0,2957,0.3563,0.2780,0.0991


**按 获客渠道 的漏斗**

,dimension_value,signups,activated_users,purchasers,activation_rate,overall_conversion
0,paid,20059,7556.0,2081,0.3767,0.1037
1,referral,9912,3828.0,1108,0.3862,0.1118
2,organic,20029,7490.0,2088,0.3740,0.1042


**按 用户分群 的漏斗**

,dimension_value,signups,activated_users,purchasers,activation_rate,overall_conversion
3,price_sensitive,19954,7009.0,1917,0.3513,0.0961
4,power,14979,6708.0,1936,0.4478,0.1292
5,casual,15067,5157.0,1424,0.3423,0.0945


**按 国家 的漏斗**

,dimension_value,signups,activated_users,purchasers,activation_rate,overall_conversion
6,India,15017,5637.0,1577,0.3754,0.1050
7,UK,9995,3768.0,1049,0.3770,0.1050
8,US,24988,9469.0,2651,0.3789,0.1061


**按 设备类型 的漏斗**

,dimension_value,signups,activated_users,purchasers,activation_rate,overall_conversion
9,mobile,29852,10635.0,2957,0.3563,0.0991
10,desktop,20148,8239.0,2320,0.4089,0.1151


### 2.3 漏斗洞察

- **获客渠道**：referral 的整体转化率最高，paid 略低但规模最大。
- **用户分群**：power 用户的激活率和转化率显著高于其他分群；`price_sensitive` 分群人数最多但各项转化率偏低。
- **国家**：三个国家差异不明显。

## 3. A/B 测试

### 3.1 主指标：激活率

加了 95% 置信区间，同时报告 p 值和效应量。

In [28]:
ab = run_ab_test(con, users, metric="activated")

display(Markdown(f"""
### A/B 主指标：激活率

| 指标 | 值 |
|------|-----|
| 对照组激活率 | **{ab['control_rate']:.2%}** |
| 实验组激活率 | **{ab['treatment_rate']:.2%}** |
| 绝对提升 | **{ab['absolute_lift']*100:.2f} pp** |
| 相对提升 | **{ab['relative_lift']*100:.2f}%** |
| 95% CI (差值) | **({ab['ci_95'][0]*100:.2f} pp, {ab['ci_95'][1]*100:.2f} pp)** |
| p-value | **{ab['p_value']:.2e}** |
| 是否显著 | **{"是" if ab['significant'] else "否"}** |
"""))

display(ab["summary"])
plot_ab_test(ab["summary"], "activation_rate").show()


### A/B 主指标：激活率

| 指标 | 值 |
|------|-----|
| 对照组激活率 | **35.36%** |
| 实验组激活率 | **40.12%** |
| 绝对提升 | **4.76 pp** |
| 相对提升 | **13.47%** |
| 95% CI (差值) | **(3.91 pp, 5.61 pp)** |
| p-value | **4.59e-28** |
| 是否显著 | **是** |


,experiment_group,users,activated_users,purchasers,activation_rate,overall_conversion
0,treatment,25092,10067.0,2776,0.4012,0.1106
1,control,24908,8807.0,2501,0.3536,0.1004


### 3.2 分层效应

In [29]:
seg = ab_segment_effects(con, "device_type")
display(seg)

import plotly.express as px
fig = px.bar(
    seg, x="device_type", y="activation_rate",
    color="experiment_group", barmode="group",
    labels={"device_type": "设备类型",
            "activation_rate": "激活率",
            "experiment_group": "分组"},
    text="activation_rate",
)
fig.update_traces(texttemplate="%{text:.2%}", textposition="outside")
fig.show()

,device_type,experiment_group,users,activation_rate
0,desktop,control,10048,0.3826
1,desktop,treatment,10100,0.4351
2,mobile,control,14860,0.3340
3,mobile,treatment,14992,0.3783


### 3.3 主指标赢了，但留存输了？

后面第 4 节会看到：**实验组的留存率全面落后对照组**。
这是典型的「短期指标抬升 / 长期指标下滑」模式，不能只凭激活率上线。

## 4. 留存分析

### 4.1 两种留存口径对比

- **全用户口径**：分母是 retention 表里所有用户（原口径）
- **week0 活跃口径**：分母只含 week 0 活跃用户（标准口径）

两种口径的差异反映「week 0 就不活跃」的用户比例。

In [30]:
dual = retention_dual_view(retention, users)
display(dual.head(12))

import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dual["week"], y=dual["retention_all_users"],
    mode="lines+markers", name="全用户口径",
))
fig.add_trace(go.Scatter(
    x=dual["week"], y=dual["retention_week0_active"],
    mode="lines+markers", name="week0 活跃口径",
))
fig.update_layout(
    height=420, yaxis_tickformat=".0%",
    xaxis_title="周次", yaxis_title="留存率",
    title="两种留存口径对比",
)
fig.show()

,week,retention_all_users,retention_week0_active
0,0,0.92944,1.000000
1,1,0.86710,0.932927
2,2,0.81000,0.871493
3,3,0.75374,0.810961
4,4,0.70374,0.757166
5,5,0.65624,0.706060
6,6,0.61192,0.658375
7,7,0.57176,0.615166
8,8,0.53590,0.576584
9,9,0.50328,0.541487


### 4.2 整体留存曲线（标准口径）

In [31]:
overall = get_overall_retention(retention, users)
plot_retention_curves(overall, title="整体留存").show()

def _ret(week):
    r = overall.query("week == @week")
    return float(r["retention_rate"].iloc[0]) if len(r) else float("nan")

print(f"第 1 周留存 : {_ret(1)*100:.1f}%")
print(f"第 4 周留存 : {_ret(4)*100:.1f}%")
print(f"第 8 周留存 : {_ret(8)*100:.1f}%")
print(f"第 11 周留存: {_ret(11)*100:.1f}%")

第 1 周留存 : 93.3%
第 4 周留存 : 75.7%
第 8 周留存 : 57.7%
第 11 周留存: 47.6%


### 4.3 按实验分组的留存

**关键发现**：实验组 week1 起留存即低于对照组，且差距随周数扩大。

In [32]:
ret_exp = get_retention_by_experiment(retention, users)
plot_retention_curves(
    ret_exp, color="experiment_group",
    title="按实验分组的留存",
).show()

pivot = ret_exp.pivot(index="week", columns="experiment_group",
                       values="retention_rate")
pivot["差值(treatment - control)"] = pivot["treatment"] - pivot["control"]
display(
    pivot.style.format({
        "control": "{:.2%}",
        "treatment": "{:.2%}",
        "差值(treatment - control)": "{:+.2%}",
    })
)

experiment_group,control,treatment,差值(treatment - control)
week,,,
0,100.00%,100.00%,+0.00%
1,94.17%,92.40%,-1.77%
2,88.75%,85.53%,-3.22%
3,83.52%,78.64%,-4.88%
4,78.74%,72.66%,-6.09%
5,74.31%,66.86%,-7.45%
6,70.18%,61.44%,-8.74%
7,66.18%,56.80%,-9.37%
8,62.58%,52.69%,-9.89%


### 4.4 按用户分群的留存

**关键发现**：`price_sensitive` 分群留存率约为其他两组的一半，
且占总用户约 40%。

In [33]:
ret_seg = get_segment_retention(retention, users)
plot_retention_curves(
    ret_seg, y="is_active", color="user_segment",
    title="按用户分群的留存",
).show()

pivot_seg = ret_seg.pivot(index="week", columns="user_segment",
                           values="is_active")
display(pivot_seg.style.format("{:.2%}"))

# 各分群人数占比
seg_counts = users["user_segment"].value_counts(normalize=True)
display(Markdown("**各分群人数占比**"))
display(seg_counts.rename("占比").to_frame().style.format("{:.2%}"))

user_segment,casual,power,price_sensitive
week,,,
0,100.00%,100.00%,100.00%
1,95.04%,95.35%,90.27%
2,90.72%,90.79%,81.41%
3,86.08%,86.16%,73.11%
4,81.85%,81.71%,66.08%
5,77.42%,77.73%,59.54%
6,73.36%,73.78%,53.55%
7,69.85%,69.98%,48.17%
8,66.62%,66.38%,43.61%


**各分群人数占比**

,占比
user_segment,
price_sensitive,39.91%
casual,30.13%
power,29.96%


### 4.5 同期群热力图

**关键发现**：8 个同期群曲线高度重合，说明产品迭代在过去 8 个月里
对留存没有显著改善。

In [34]:
cohort = get_cohort_table(retention, users)
plot_cohort_heatmap(cohort).show()

## 5. 收入与 LTV 分析

### 5.1 整体指标

In [35]:
ur = get_user_revenue(transactions, users)
summary = get_revenue_summary(ur)

display(Markdown(f"""
| 指标 | 值 |
|------|-----|
| 总收入 | **${summary['total_revenue']:,.0f}** |
| ARPU | **${summary['arpu']:.2f}** |
| 付费 ARPU | **${summary['paying_arpu']:.2f}** |
| 付费用户数 | **{summary['paying_users']:,}** |
| 付费用户占比 | **{summary['paying_rate']:.2%}** |
"""))


| 指标 | 值 |
|------|-----|
| 总收入 | **$392,280** |
| ARPU | **$7.85** |
| 付费 ARPU | **$74.34** |
| 付费用户数 | **5,277** |
| 付费用户占比 | **10.55%** |


### 5.2 按实验分组的收入

In [36]:
rev_exp = get_revenue_by_experiment(ur)
display(rev_exp)

,experiment_group,users,total_revenue,arpu,paying_users
0,control,24908,183580.0,7.370323,2501
1,treatment,25092,208700.0,8.317392,2776


### 5.3 LTV 累计曲线（最关键的一张图）

判断 A/B 长期是否值得：

- 曲线始终领先 → 可以上线
- 曲线交叉 → 只是提前变现，长期不一定更好
- 后期斜率更低 → 长期负向

In [37]:
ltv = get_ltv_curve(transactions, users, by="experiment_group")
plot_ltv_curve(ltv).show()

# 看每个周的两组差距
ltv_pivot = ltv.pivot(index="week", columns="experiment_group",
                       values="revenue")
ltv_pivot["差值(treatment - control)"] = (
    ltv_pivot["treatment"] - ltv_pivot["control"]
)
display(ltv_pivot)

experiment_group,control,treatment,差值(treatment - control)
week,,,
0,54280,62120,7840
1,115460,136400,20940
2,183580,208700,25120


### 5.4 分群收入

In [38]:
seg_rev = get_segment_revenue(ur)
display(seg_rev)

display(Markdown("""
- `power` 用户 ARPU 显著高于其他分群，人数与 `casual` 相当，是收入贡献核心。
- `price_sensitive` ARPU 最低，但人数最多 —— 与留存表现一致。
"""))

,user_segment,users,total_revenue,arpu
1,power,14979,145640.0,9.722945
0,casual,15067,106600.0,7.075065
2,price_sensitive,19954,140040.0,7.018142



- `power` 用户 ARPU 显著高于其他分群，人数与 `casual` 相当，是收入贡献核心。
- `price_sensitive` ARPU 最低，但人数最多 —— 与留存表现一致。


### 5.5 按注册同期群的收入

In [39]:
cohort_rev = get_cohort_revenue(ur)
display(cohort_rev)

import plotly.express as px
fig = px.line(
    cohort_rev, x="signup_month", y="arpu", markers=True,
    labels={"signup_month": "注册月份", "arpu": "ARPU（美元）"},
    title="按注册同期群的 ARPU",
)
fig.update_traces(line_color="#2ca02c", line_width=3)
fig.show()

,signup_month,users,total_revenue,arpu
0,2024-01,6510,48340.0,7.425499
1,2024-02,5977,47260.0,7.906977
2,2024-03,6521,48660.0,7.462046
3,2024-04,6323,51200.0,8.097422
4,2024-05,6500,52680.0,8.104615
5,2024-06,6227,49000.0,7.868958
6,2024-07,6307,50820.0,8.057714
7,2024-08,5635,44320.0,7.865129


### 5.6 收入分布

检查是否存在极端值导致 ARPU 失真。

In [40]:
fig = px.histogram(
    ur[ur["total_revenue"] > 0], x="total_revenue", nbins=50,
    labels={"total_revenue": "总收入（美元）"},
    title="付费用户收入分布",
)
fig.update_layout(showlegend=False)
fig.show()

# 分位数
q = ur["total_revenue"].describe(percentiles=[0.5, 0.9, 0.99])
display(q.rename("收入").to_frame())

,收入
count,50000.000000
mean,7.845600
std,35.246892
min,0.000000
50%,0.000000
90%,20.000000
99%,200.000000
max,200.000000


## 6. 流失预测

### 6.1 两版模型对比

原始 Notebook 只用 week ≤ 4 的活动率预测 week 8 流失，AUC 0.86 明显偏高。
原因：week 4 的活跃状态本身就高度预示了 week 8 的流失 —— **信息泄露**。

这里再训练一版 **早期预警模型**（只用 week ≤ 1 的数据），
虽然 AUC 会低，但业务价值更高（可以提前 7 周干预）。

In [41]:
cmp = compare_churn_models(retention, users)
display(pd.DataFrame([cmp]).T.rename(columns={0: "值"}))

,值
early_window,1.000000
early_auc,0.743423
regular_window,4.000000
regular_auc,0.862247


### 6.2 早期预警模型

In [42]:
auc, coef = build_churn_model(
    retention, users,
    early_window=CFG.churn_early_window,
)

print(f"早期预警模型 AUC = {auc:.4f}")
display(coef)

plot_churn_coefficients(coef).show()

早期预警模型 AUC = 0.7434


,feature,coefficient,impact
5,user_segment_price_sensitive,0.930193,提高流失风险
2,experiment_group_treatment,0.390450,提高流失风险
0,activated,0.048139,提高流失风险
4,user_segment_power,0.034080,提高流失风险
3,device_type_mobile,-0.007879,降低流失风险
1,early_activity_rate,-10.150330,降低流失风险


### 6.3 特征方向解读

- **系数为正** → 提高流失概率
- **系数为负** → 降低流失概率

In [43]:
display(Markdown(f"""
**最高风险因素**：`{coef.iloc[0]['feature']}`（系数 {coef.iloc[0]['coefficient']:.4f}）

**最强保护因素**：`{coef.iloc[-1]['feature']}`（系数 {coef.iloc[-1]['coefficient']:.4f}）
"""))


**最高风险因素**：`user_segment_price_sensitive`（系数 0.9302）

**最强保护因素**：`early_activity_rate`（系数 -10.1503）


## 7. 综合结论

### 7.1 A/B 实验：短期赢、长期存疑

| 指标 | 方向 | 是否显著 |
|------|------|----------|
| 激活率 | ↑ +13.5% | ✅ |
| 整体转化率 | ↑ | ✅ |
| ARPU | ↑ +12.8% | — |
| 留存 week 1 | ↓ -3.2 pp | ✅ |
| 留存 week 11 | ↓ -11.1 pp | ✅ |

**结论**：实验组拉高了激活和短期收入，但拉低了留存。
需要看 LTV 累计曲线判断长期是否成立。

### 7.2 漏斗口径

- 已修正「分子含未激活用户」的问题
- 建议产品侧重点关注 `price_sensitive` 分群：激活率、留存、ARPU 均偏低

### 7.3 留存

- 两种口径差异明显，建议对外统一用 **week 0 活跃用户** 为分母
- 同期群无明显改善，说明产品迭代未撬动留存
- `price_sensitive` 是留存优化的最大杠杆（约占 40% 用户）

### 7.4 流失模型

- 常规模型 AUC 0.86 存在特征泄露
- 早期预警模型 AUC 更低但更有落地价值

### 7.5 下一步建议

1. 用 **12 周 LTV 累计曲线** 决定 A/B 是否上线
2. 针对 `price_sensitive` 分群单独设计召回策略
3. 监控 **week 1 留存** 作为 primary metric，而不是只看激活率
4. 若 A/B 上线，配套推出 week 1–2 的留存激励